# Archetype clustering algorithm comparison

This notebook compares several clustering methods on **optimal HP config vectors**. **k is not fixed**: each algorithm searches over k (or its own parameters) for **peak performance**, then we compare which achieves the best overall.

**Clustering methods compared:**
1. **K-Means** — Partitional; good for convex, spherical clusters; grid search over k for best.
2. **GMM** — Soft assignment, ellipsoidal clusters; grid search over k for best.
3. **Ward (Agglomerative)** — Merge by minimum variance; grid search over k for best.
4. **Spectral Clustering** — Similarity-graph based, good for non-convex shapes; grid search over k for best.
5. **MiniBatch K-Means** — Fast approximate K-Means; grid search over k for best.
6. **DBSCAN** — Density-based, no fixed k, can find noise; grid search over `eps` and `min_samples` for best.

**Metrics:** Silhouette (higher better), Calinski-Harabasz (higher better), Davies-Bouldin (lower better). A composite score selects each algorithm’s peak and the final ranking.

## 1. Configuration and dependencies

In [7]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.cluster import SpectralClustering
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score

HP_DB_PATH = "hp_tuning_db_refined.csv"
RANDOM_STATE = 42
# k search range (for K-Means / GMM / Ward / Spectral / MiniBatch)
K_MIN, K_MAX = 2, 12

from train_hp_meta_model import HP_PARAM_NAMES
print("Dependencies loaded.")

依赖加载完成。


## 2. Prepare data (aligned with compute_archetypes in train)

In [8]:
df = pd.read_csv(HP_DB_PATH)
if 'dataset_name' not in df.columns or 'primary_score' not in df.columns:
    raise ValueError("CSV must contain dataset_name and primary_score")

best_mask = df.groupby('dataset_name')['primary_score'].transform('max') == df['primary_score']
best_rows = df[best_mask].drop_duplicates(subset='dataset_name', keep='first').copy()
print(f"One best config per dataset: {len(best_rows)} rows")

hp_cols = [c for c in HP_PARAM_NAMES if c in best_rows.columns]
X_hp = best_rows[hp_cols].copy()

for col in hp_cols:
    param = col.replace('hp_', '')
    if param in ['learning_rate', 'n_estimators', 'reg_alpha', 'reg_lambda']:
        X_hp[col] = np.log1p(X_hp[col].clip(lower=1e-10))

X_hp = X_hp.fillna(X_hp.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_hp)
X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)

n_samples = len(X_scaled)
k_max_actual = min(K_MAX, max(K_MIN, n_samples // 10))
print(f"Feature dim: {X_scaled.shape[1]}, samples: {n_samples}, k search range: [{K_MIN}, {k_max_actual}]")

每个 dataset 取最优配置: 1749 行
特征维度: 10, 样本数: 1749, k 搜索范围: [2, 12]


## 3. Define clustering algorithms and evaluation

In [9]:
def run_kmeans(X, k, seed=42):
    m = KMeans(n_clusters=k, random_state=seed, n_init=10)
    return m.fit_predict(X)

def run_gmm(X, k, seed=42):
    m = GaussianMixture(n_components=k, random_state=seed, n_init=3, max_iter=200)
    m.fit(X)
    return m.predict(X)

def run_ward(X, k):
    m = AgglomerativeClustering(n_clusters=k, linkage='ward')
    return m.fit_predict(X)

def run_spectral(X, k, seed=42):
    m = SpectralClustering(n_clusters=k, affinity='nearest_neighbors', n_neighbors=min(10, len(X)//max(1,k)), random_state=seed)
    return m.fit_predict(X)

def run_minibatch_kmeans(X, k, seed=42):
    m = MiniBatchKMeans(n_clusters=k, random_state=seed, n_init=3, batch_size=256)
    return m.fit_predict(X)

def run_dbscan(X, eps, min_samples=5):
    m = DBSCAN(eps=eps, min_samples=min_samples)
    return m.fit_predict(X)

def labels_without_noise_for_metrics(X, labels):
    """Assign DBSCAN noise points (-1) to nearest cluster so we get a full partition for Silhouette/CH/DB."""
    labels = np.asarray(labels, dtype=int)
    noise_mask = labels == -1
    if not np.any(noise_mask):
        return labels
    n_clusters = len(np.unique(labels[~noise_mask]))
    if n_clusters == 0:
        return labels
    from sklearn.metrics.pairwise import euclidean_distances
    X_noise = X[noise_mask]
    X_core = X[~noise_mask]
    L_core = labels[~noise_mask]
    centers = np.array([X_core[L_core == c].mean(axis=0) for c in np.unique(L_core)])
    d = euclidean_distances(X_noise, centers)
    out = labels.copy()
    out[noise_mask] = np.unique(L_core)[np.argmin(d, axis=1)]
    return out

def evaluate_labels(X, labels, k=None):
    labels = np.asarray(labels)
    n_unique = len(np.unique(labels[labels >= 0]))
    if n_unique < 2:
        return {'silhouette': np.nan, 'calinski_harabasz': np.nan, 'davies_bouldin': np.nan}
    try:
        sil = silhouette_score(X, labels)
    except Exception:
        sil = np.nan
    try:
        ch = calinski_harabasz_score(X, labels)
    except Exception:
        ch = np.nan
    try:
        db = davies_bouldin_score(X, labels)
    except Exception:
        db = np.nan
    return {'silhouette': sil, 'calinski_harabasz': ch, 'davies_bouldin': db}

def composite_score(metrics):
    s = np.nan_to_num(metrics['silhouette'], nan=0.0)
    ch = np.nan_to_num(metrics['calinski_harabasz'], nan=0.0)
    db = np.nan_to_num(metrics['davies_bouldin'], nan=1e6)
    return float(s), float(ch), float(db)

print("Clustering and evaluation functions defined.")

聚类与评估函数已定义。


## 4. Search peak for each algorithm and record metrics (best k/params per algorithm)

In [10]:
results = []
k_range = range(K_MIN, k_max_actual + 1)

def pick_best(candidates, key_fn=None):
    if key_fn is None:
        key_fn = lambda r: (np.nan_to_num(r['silhouette']), np.nan_to_num(r['calinski_harabasz']), -np.nan_to_num(r['davies_bouldin'], nan=1e6))
    return max(candidates, key=key_fn)

# ---------- Algorithms that search over k ----------
for k in k_range:
    try:
        labels = run_kmeans(X_scaled, k)
        metrics = evaluate_labels(X_scaled, labels)
        metrics['algorithm'] = 'K-Means'
        metrics['best_param'] = f'k={k}'
        metrics['n_clusters_actual'] = len(np.unique(labels))
        metrics['n_noise'] = 0
        results.append(metrics)
    except Exception:
        pass
best_kmeans = pick_best([r for r in results if r['algorithm'] == 'K-Means'])

for k in k_range:
    try:
        labels = run_gmm(X_scaled, k)
        metrics = evaluate_labels(X_scaled, labels)
        metrics['algorithm'] = 'GMM'
        metrics['best_param'] = f'k={k}'
        metrics['n_clusters_actual'] = len(np.unique(labels))
        metrics['n_noise'] = 0
        results.append(metrics)
    except Exception:
        pass
best_gmm = pick_best([r for r in results if r['algorithm'] == 'GMM'])

for k in k_range:
    try:
        labels = run_ward(X_scaled, k)
        metrics = evaluate_labels(X_scaled, labels)
        metrics['algorithm'] = 'Ward (Agglomerative)'
        metrics['best_param'] = f'k={k}'
        metrics['n_clusters_actual'] = len(np.unique(labels))
        metrics['n_noise'] = 0
        results.append(metrics)
    except Exception:
        pass
best_ward = pick_best([r for r in results if r['algorithm'] == 'Ward (Agglomerative)'])

for k in k_range:
    try:
        labels = run_spectral(X_scaled, k)
        metrics = evaluate_labels(X_scaled, labels)
        metrics['algorithm'] = 'Spectral Clustering'
        metrics['best_param'] = f'k={k}'
        metrics['n_clusters_actual'] = len(np.unique(labels))
        metrics['n_noise'] = 0
        results.append(metrics)
    except Exception:
        pass
best_spectral = pick_best([r for r in results if r['algorithm'] == 'Spectral Clustering'])

for k in k_range:
    try:
        labels = run_minibatch_kmeans(X_scaled, k)
        metrics = evaluate_labels(X_scaled, labels)
        metrics['algorithm'] = 'MiniBatch K-Means'
        metrics['best_param'] = f'k={k}'
        metrics['n_clusters_actual'] = len(np.unique(labels))
        metrics['n_noise'] = 0
        results.append(metrics)
    except Exception:
        pass
best_mbk = pick_best([r for r in results if r['algorithm'] == 'MiniBatch K-Means'])

# ---------- DBSCAN: grid search over eps, min_samples ----------
eps_grid = np.linspace(0.15, 2.5, 20)
min_samples_list = [2, 5, 10]
dbscan_candidates = []
for eps in eps_grid:
    for ms in min_samples_list:
        try:
            labels = run_dbscan(X_scaled, eps=float(eps), min_samples=ms)
            n_clusters = len(np.unique(labels[labels >= 0]))
            n_noise = int((labels == -1).sum())
            if n_clusters < 2 or n_noise / max(1, len(labels)) > 0.6:
                continue
            labels_eval = labels_without_noise_for_metrics(X_scaled, labels)
            metrics = evaluate_labels(X_scaled, labels_eval)
            metrics['algorithm'] = 'DBSCAN'
            metrics['best_param'] = f'eps={eps:.2f}, min_samples={ms}'
            metrics['n_clusters_actual'] = n_clusters
            metrics['n_noise'] = n_noise
            dbscan_candidates.append(metrics)
        except Exception:
            pass
if dbscan_candidates:
    best_dbscan = pick_best(dbscan_candidates)
else:
    best_dbscan = {'algorithm': 'DBSCAN', 'best_param': '-', 'n_clusters_actual': 0, 'n_noise': 0, 'silhouette': np.nan, 'calinski_harabasz': np.nan, 'davies_bouldin': np.nan}

# ---------- Keep only peak result per algorithm ----------
peak_results = [best_kmeans, best_gmm, best_ward, best_spectral, best_mbk, best_dbscan]
res_df = pd.DataFrame(peak_results)
print("Peak results per algorithm:")
for _, r in res_df.iterrows():
    extra = f", n_noise={r.get('n_noise', 0)}" if r.get('n_noise', 0) else ""
    print(f"  {r['algorithm']} ({r['best_param']}): k={r['n_clusters_actual']}{extra}, Sil={r['silhouette']:.4f}, CH={r['calinski_harabasz']:.2f}, DB={r['davies_bouldin']:.4f}")

各算法峰值结果:
  K-Means (k=6): k=6, Sil=0.1508, CH=204.68, DB=1.8812
  GMM (k=2): k=2, Sil=0.0668, CH=94.53, DB=4.2055
  Ward (Agglomerative) (k=2): k=2, Sil=0.2507, CH=180.89, DB=1.7157
  Spectral Clustering (k=12): k=12, Sil=0.0162, CH=107.42, DB=1.8198
  MiniBatch K-Means (k=8): k=8, Sil=0.1409, CH=175.30, DB=1.8160
  DBSCAN (eps=2.38, min_samples=10): k=2, n_noise=70, Sil=0.2988, CH=127.97, DB=1.3691


## 5. Metrics table and ranking

In [11]:
cols_display = ['algorithm', 'best_param', 'n_clusters_actual', 'silhouette', 'calinski_harabasz', 'davies_bouldin']
if 'n_noise' in res_df.columns:
    cols_display.append('n_noise')
display(res_df[[c for c in cols_display if c in res_df.columns]])

# Composite score: higher Silhouette and CH better, lower DB better → after normalization, higher Sil+CH+norm(1-DB) is better
s = res_df['silhouette'].fillna(0)
ch = res_df['calinski_harabasz'].fillna(0)
db = res_df['davies_bouldin'].fillna(1e6)
if ch.max() > ch.min():
    ch_norm = (ch - ch.min()) / (ch.max() - ch.min())
else:
    ch_norm = pd.Series(0.5, index=ch.index)
if db.max() > db.min():
    db_norm = 1 - (db - db.min()) / (db.max() - db.min())
else:
    db_norm = pd.Series(0.5, index=db.index)
res_df['score'] = s + ch_norm + db_norm
res_df = res_df.sort_values('score', ascending=False).reset_index(drop=True)
print("\nRanked by composite score (Silhouette + normalized CH + normalized(1-DB)):")
display(res_df[['algorithm', 'best_param', 'n_clusters_actual', 'silhouette', 'calinski_harabasz', 'davies_bouldin', 'score']])

,algorithm,best_param,n_clusters_actual,silhouette,calinski_harabasz,davies_bouldin,n_noise
0,K-Means,k=6,6,0.150782,204.683730,1.881234,0
1,GMM,k=2,2,0.066825,94.533355,4.205477,0
2,Ward (Agglomerative),k=2,2,0.250738,180.888308,1.715709,0
3,Spectral Clustering,k=12,12,0.016207,107.419603,1.819815,0
4,MiniBatch K-Means,k=8,8,0.140934,175.301014,1.816030,0
5,DBSCAN,"eps=2.38, min_samples=10",2,0.298789,127.970724,1.369073,70



按综合得分排序 (Silhouette + 归一化CH + 归一化(1-DB)):


,algorithm,best_param,n_clusters_actual,silhouette,calinski_harabasz,davies_bouldin,score
0,K-Means,k=6,6,0.150782,204.683730,1.881234,1.970215
1,Ward (Agglomerative),k=2,2,0.250738,180.888308,1.715709,1.912502
2,MiniBatch K-Means,k=8,8,0.140934,175.301014,1.816030,1.716604
3,DBSCAN,"eps=2.38, min_samples=10",2,0.298789,127.970724,1.369073,1.602350
4,Spectral Clustering,k=12,12,0.016207,107.419603,1.819815,0.974281
5,GMM,k=2,2,0.066825,94.533355,4.205477,0.066825


## 6. Best clustering algorithm: performance and summary

In [12]:
best = res_df.iloc[0]
best_name = best['algorithm']
best_param = best.get('best_param', '-')

print("=" * 60)
print("Best clustering algorithm (peak comparison at each algorithm's best k/params)")
print("=" * 60)
print(f"  Algorithm: {best_name}")
print(f"  Best params: {best_param}")
print(f"  Clusters: {best.get('n_clusters_actual', '-')}", end="")
if best.get('n_noise', 0):
    print(f", noise points: {best['n_noise']}")
else:
    print()
print(f"  Silhouette:       {best['silhouette']:.4f}  (higher better, [-1,1])")
print(f"  Calinski-Harabasz: {best['calinski_harabasz']:.2f}  (higher better)")
print(f"  Davies-Bouldin:   {best['davies_bouldin']:.4f}  (lower better)")
print(f"  Composite score:  {best['score']:.4f}")
print("=" * 60)
print("\nSummary:")
print(f"  On {n_samples} datasets' optimal HP vectors, each algorithm searched for its peak (k or eps/min_samples); {best_name} has the best peak performance.")
print("  You can replace KMeans in train_hp_meta_model.compute_archetypes with this algorithm (using the best params above) and retrain, or keep the current K-Means as default.")
print("  Peak ranking:")
for i, row in res_df.iterrows():
    p = row.get('best_param', '')
    print(f"    {i+1}. {row['algorithm']} ({p})  score={row['score']:.4f}")

最优聚类算法（各算法在各自最优 k/参数下的峰值对比）
  算法: K-Means
  最优参数: k=6
  簇数: 6
  Silhouette:       0.1508  (越大越好，[-1,1])
  Calinski-Harabasz: 204.68  (越大越好)
  Davies-Bouldin:   1.8812  (越小越好)
  综合得分:         1.9702

总结:
  在 1749 个 dataset 的最优 HP 向量上，各算法在各自特长下搜索峰值（k 或 eps/min_samples）后，K-Means 的峰值表现最佳。
  可将 train_hp_meta_model.compute_archetypes 中的 KMeans 替换为该算法（并采用其上表最优参数）后重新训练，或保留当前 K-Means 作为默认。
  各算法峰值排名:
    1. K-Means (k=6)  score=1.9702
    2. Ward (Agglomerative) (k=2)  score=1.9125
    3. MiniBatch K-Means (k=8)  score=1.7166
    4. DBSCAN (eps=2.38, min_samples=10)  score=1.6023
    5. Spectral Clustering (k=12)  score=0.9743
    6. GMM (k=2)  score=0.0668
